Memory Management
-----------------

Let's start by creating a large enough set of data that you'll see it if you're watching your memory usage.  Separately from this notebook, open a console and start `htop` to keep an eye on how much RAM your notebook is using.

In [ ]:
import numpy as np

Note down how much RAM this notebook is using (in the VIRT column), likely something like 950GB before creating any data.

In [ ]:
data = np.random.randn(100000000)

This array takes up about 800MB of RAM.  Now let's see what kinds of operations on this array might increase the amount of memory this notebook is using.  Let's start with some in-place arithmetic.  This should multiply each element of `data` by 2, "in place", meaning it doesn't allocate an entire new array to do the work.

In [ ]:
data *= 2

As expected, this doesn't increase the memory footprint.  What if we did this operation without the inplace `+=` operator?

In [ ]:
data = data + 2

It turns out, Python is smart enough to run this operation in-place too!  What if we store the result in a new variable?

In [ ]:
data2 = data * 2

You'll notice that your memory usage has now increased by another 800MB.  Often you don't need to keep these kinds of intermediate variables.  We can use the `del` operation to remove variables from memory.

In [ ]:
del data, data2

Sometimes you'll find that memory that you expect to have been released is still in use.  One way to try to clean this up is to use the `gc` module.

In [ ]:
import gc
gc.collect()

Vectorization
-------------

Let's try something more complex.  Let's create two large arrays and do some matrix operations with them.

In [ ]:
m = np.random.randn(1000000)
v = np.random.randn(100)

How do we multiply the transpose of `m` by `v` to create a matrix of size `(1000000, 100)`?  One way is to write a very long loop in Python:

In [ ]:
%%time
d = np.zeros((m.size, v.size))
for i in range(m.size):
    for j in range(v.size):
        d[i, j] = m[i] * v[j]

You'll notice that this operation takes a very long time.  What's a way to make this faster?  Perhaps we can multiply `m` by one of the values in `v` in a loop, then concatenate them all together into an array.

In [ ]:
%%time
d = [m * vv for vv in v]

In [ ]:
%%time
d = np.array([m * vv for vv in v])

Notice that creating the array from the list takes about as long as the multiplication operation itself.  What if we'd done the loop over the larger `m` array?

In [ ]:
%%time
d = np.array([mm * v for mm in m])

There is a way to make this operation very efficient, so that the array creation and multiplication all happen in `C` under the hood:

In [ ]:
%%time
d = m[:, None] * v

This manipulation of the `m` array is known as [broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html).  It's a very powerful trick for performing optimized vector operations, where the meat of the work is done in `C` instead of in Python.  The numpy documentation has some very thorough tutorials on how to use this wisely.

Essentially what's happening here is that we are telling numpy to treat `m` as a 2D array, where each row is a copy of the 1D array, and the size of the second dimension simply matches that of the relevant dimension of `v`.  This means that this operation is performed like the multiplication of a 1000000x100 matrix `m` against a vector `v` of length 100.  The end result is a 2D array of shape:

In [ ]:
d.shape

SPT Map Manipulation
--------------------

We typically store maps for SPT in `.g3` files that contain `G3Frame` objects.  See the spt3g_software [documentation](https://southpoletelescope.github.io/spt3g_software/frames.html) and the other tutorial in this directory for details.  Here, I'll assume that you already know how to read frames from disk.  For details on map objects and the various pipeline modules described here, see [this page](https://southpoletelescope.github.io/spt3g_software/moddoc_maps.html).

In [ ]:
import sys
sys.path.insert(0, "/home/arahlin/code/spt3g_software/build")

In [ ]:
import numpy as np
from spt3g import core, cluster, maps

I'm going to be doing some operations on data that's stored on the grid, so I'll use the grid-friendly tools, which first copy each file to a local disk.  To do that, I'll need to make sure I have an active grid token.  The following command either returns the path for an active token, or raises an error if your token has expired or is missing.  This is essentially equivalent to running `token-info` on the command line.

In [ ]:
cluster.get_grid_token()

Let's load a random winter field observation from disk.  I know that this file contains several other frames in it, so I'm going to use this shortcut to keep just the last frame, which I know contains the map we want:

In [ ]:
frame = list(cluster.GridFile("/sptgrid/data/onlinemaps/ra0hdec-44.75/294349321_150GHz_tonly.g3.gz"))[-1]

In [ ]:
print(frame)

This is a typical map frame containing some temperature-only data.  Data are stored with weights applied, because that's how they're constructed in mapmaking, and because the operation of removing weights can be lossy for poorly conditioned polarization data.  Here we're just dealing with temperature data, so these operations are greatly simplified.

Let's take a look at the temperature map in the frame:

In [ ]:
m = frame["T"]
print(m)
print("Shape:", m.shape)
print("Size:", m.size)
print("Allocated:", m.npix_allocated)
print("Fraction allocated:", m.npix_allocated / m.size)
print("Sparse?", m.sparse)

Notice that this map is sparsely populated -- only 18% of the pixels in the map are "allocated", meaning that they are filled with data that is likely non-zero.  This is a memory-efficient way of storing what is otherwise a pretty large 2D array.  You'll want to be careful with what kinds of operations you do on this array to avoid increasing the memory usage significantly.

One of the first operations we would do on such a frame is to remove the weights from the Stokes maps (T, and Q/U if present).  We do this with a canned function* that takes a frame as an input argument.  We use the `zero_nans` argument to avoid populating all of the currently unallocated pixels with `NaN` values:

In [ ]:
maps.RemoveWeights(frame, zero_nans=True)

Notice that our overall memory usage hasn't increased since we loaded the map into memory.  Let's plot this map up:

In [ ]:
from matplotlib import pyplot as plt
%matplotlib inline

In [ ]:
plt.imshow(m, vmin=-200 * core.G3Units.uK, vmax=200 * core.G3Units.uK);

Notice that our memory usage has gone up significantly.  Why?  Well, this plotting function has implicitly constructed a "dense" numpy array out of the map data.

In [ ]:
print(m.sparse)
print("Fraction allocated:", m.npix_allocated / m.size)

Let's turn this back into a compact map:

In [ ]:
m.compact()
print(m.sparse)
print("Fraction allocated:", m.npix_allocated / m.size)

... and do some additional garbage collection (likely something internal to matplotlib)

In [ ]:
import gc
gc.collect()

We can also add two maps together, provided that they are both built on the same "footprint" or "stub", i.e. they use the same coordinate transformation.  For example, let's add another subfield to our map:

In [ ]:
frame2 = list(cluster.GridFile("/sptgrid/data/onlinemaps/ra0hdec-52.25/294249241_150GHz_tonly.g3.gz"))[-1]
maps.RemoveWeights(frame2, zero_nans=True)
coadd = m + frame2["T"]

Notice that this coadded map is also sparsely sampled, so this map addition operation preserves the sparsity of the input map.

In [ ]:
print(coadd.sparse)
print(coadd.npix_allocated / coadd.size)

In [ ]:
plt.imshow(coadd, vmin=-200 * core.G3Units.uK, vmax=200 * core.G3Units.uK);

In [ ]:
coadd.compact()
gc.collect()

All of the typical arithmetic operations that one can do with map objects have been written in a way that tries to preserve the sparsity of the map as much as possible.  Map objects also have methods for most of the common numpy operations that one might do on an array, which can optionally avoid zeroes (empty pixels) and/or NaN values.  See the documentation for details.  The maps [tests directory](https://github.com/SouthPoleTelescope/spt3g_software/tree/master/maps/tests) also contains several scripts for testing various functionality so you can see how many of the methods and functions can be used there.